# Autonomous Financial Analysis and Ratio Extraction Agent with Tool Use

In financial analysis, professionals routinely synthesize two distinct types of data:
1. **Quantitative metrics**: Exact financial statement figures (Revenue, Net Income, EBITDA, Debt) and calculated valuation multiples (P/E, Debt-to-Equity, Operating Margins).
2. **Qualitative context**: Sector dynamics, competitive moats, market sentiment, and macroeconomic headwinds.

While Large Language Models excel at synthesizing qualitative narratives, delegating mathematical arithmetic to LLM generation can lead to calculation errors and hallucinated multiples. 

In this cookbook recipe, we demonstrate how to build an **autonomous financial research agent** using Claude (`claude-sonnet-5`) and **client-side tools**. The agent:
- Queries market data tools to extract audited balance sheet and income statement metrics.
- Executes deterministic mathematical tools to compute essential valuation and leverage ratios without arithmetic hallucination.
- Retrieves industry benchmark multiples for comparative peer analysis.
- Synthesizes a structured equity research memo with tables, solvency assessments, and key investment takeaways.

## Step 1: Set up the environment

First, install the required libraries and initialize the Anthropic client. We use `claude-sonnet-5` for strong reasoning and reliable tool calling.

In [1]:
%pip install -q anthropic pydantic pandas

In [2]:
import os
from anthropic import Anthropic

# Ensure your ANTHROPIC_API_KEY environment variable is set
client = Anthropic()
MODEL_NAME = "claude-sonnet-5"

## Step 2: Define Financial Data and Client-Side Tool Implementations

We provide Claude with three client-side functions:
- `get_company_financials`: Returns reported financial statement figures for a given ticker.
- `calculate_financial_ratios`: Executes deterministic math for P/E, Net Debt, Debt-to-Equity, and EBITDA margin.
- `get_industry_benchmarks`: Retrieves peer sector averages for relative valuation comparison.

In [3]:
from typing import Any, Dict

# Mock financial repository representing latest audited annual filings
FINANCIAL_DATABASE = {
    "NVDA": {
        "name": "Nvidia Corporation",
        "sector": "Semiconductors",
        "revenue_ttm": 126_000_000_000,
        "net_income_ttm": 63_000_000_000,
        "ebitda_ttm": 72_000_000_000,
        "total_debt": 11_000_000_000,
        "cash_and_equivalents": 34_800_000_000,
        "shareholders_equity": 58_000_000_000,
        "market_cap": 3_100_000_000_000,
    },
    "AAPL": {
        "name": "Apple Inc.",
        "sector": "Consumer Electronics",
        "revenue_ttm": 391_000_000_000,
        "net_income_ttm": 93_700_000_000,
        "ebitda_ttm": 130_000_000_000,
        "total_debt": 106_000_000_000,
        "cash_and_equivalents": 65_000_000_000,
        "shareholders_equity": 66_700_000_000,
        "market_cap": 3_400_000_000_000,
    },
}

INDUSTRY_BENCHMARKS = {
    "Semiconductors": {
        "avg_pe_ratio": 35.5,
        "avg_ebitda_margin_pct": 38.0,
        "avg_debt_to_equity": 0.45,
    },
    "Consumer Electronics": {
        "avg_pe_ratio": 28.0,
        "avg_ebitda_margin_pct": 26.5,
        "avg_debt_to_equity": 1.15,
    },
}


def get_company_financials(ticker: str) -> Dict[str, Any]:
    ticker_upper = ticker.upper()
    if ticker_upper in FINANCIAL_DATABASE:
        return FINANCIAL_DATABASE[ticker_upper]
    return {"error": f"Ticker '{ticker}' not found in database"}


def calculate_financial_ratios(
    market_cap: float,
    net_income: float,
    total_debt: float,
    cash: float,
    shareholders_equity: float,
    ebitda: float,
    revenue: float,
) -> Dict[str, Any]:
    pe_ratio = round(market_cap / net_income, 2) if net_income else None
    net_debt = total_debt - cash
    debt_to_equity = round(total_debt / shareholders_equity, 2) if shareholders_equity else None
    ebitda_margin = round(ebitda / revenue, 4) if revenue else None

    return {
        "pe_ratio": pe_ratio,
        "net_debt": net_debt,
        "net_cash_positive": net_debt < 0,
        "debt_to_equity": debt_to_equity,
        "ebitda_margin_pct": round(ebitda_margin * 100, 2) if ebitda_margin else None,
    }


def get_industry_benchmarks(sector: str) -> Dict[str, Any]:
    if sector in INDUSTRY_BENCHMARKS:
        return INDUSTRY_BENCHMARKS[sector]
    return {"error": f"Sector '{sector}' not found in benchmark table"}

## Step 3: Define the Tool Schemas for Claude

Next, we describe our tools using the Anthropic tool use schema format. Clear field descriptions help Claude choose appropriate tools and construct valid parameters.

In [4]:
tools = [
    {
        "name": "get_company_financials",
        "description": "Retrieves audited financial statement metrics for a company by ticker symbol.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol (e.g., 'NVDA', 'AAPL').",
                }
            },
            "required": ["ticker"],
        },
    },
    {
        "name": "calculate_financial_ratios",
        "description": "Deterministically calculates valuation and solvency ratios (P/E, Net Debt, Debt-to-Equity, EBITDA Margin).",
        "input_schema": {
            "type": "object",
            "properties": {
                "market_cap": {"type": "number", "description": "Market capitalization in USD"},
                "net_income": {"type": "number", "description": "Net income in USD"},
                "total_debt": {"type": "number", "description": "Total debt in USD"},
                "cash": {"type": "number", "description": "Cash and cash equivalents in USD"},
                "shareholders_equity": {"type": "number", "description": "Total shareholders' equity in USD"},
                "ebitda": {"type": "number", "description": "EBITDA in USD"},
                "revenue": {"type": "number", "description": "Total revenue in USD"},
            },
            "required": [
                "market_cap",
                "net_income",
                "total_debt",
                "cash",
                "shareholders_equity",
                "ebitda",
                "revenue",
            ],
        },
    },
    {
        "name": "get_industry_benchmarks",
        "description": "Retrieves sector benchmark valuation and profitability metrics.",
        "input_schema": {
            "type": "object",
            "properties": {
                "sector": {
                    "type": "string",
                    "description": "Sector name (e.g., 'Semiconductors', 'Consumer Electronics').",
                }
            },
            "required": ["sector"],
        },
    },
]

## Step 4: Implement the Autonomous Agent Execution Loop

The agent orchestrator sends the user prompt along with our tool definitions to Claude.
When Claude decides to invoke tools (`stop_reason == "tool_use"`), we:
1. Parse the tool calls from Claude's response.
2. Execute the corresponding local Python functions.
3. Return the results as `tool_result` content blocks.
4. Continue the loop until Claude produces its final synthesis (`stop_reason == "end_turn"`).

In [5]:
import json


def dispatch_tool(tool_name: str, tool_input: Dict[str, Any]) -> Any:
    if tool_name == "get_company_financials":
        return get_company_financials(**tool_input)
    elif tool_name == "calculate_financial_ratios":
        return calculate_financial_ratios(**tool_input)
    elif tool_name == "get_industry_benchmarks":
        return get_industry_benchmarks(**tool_input)
    else:
        return {"error": f"Unknown tool: {tool_name}"}


def run_financial_agent(user_query: str, verbose: bool = True) -> str:
    system_prompt = (
        "You are an expert equity research analyst at a global investment firm. "
        "Your mandate is to perform evidence-grounded financial assessments. "
        "Always query financial data and calculate valuation metrics deterministically using "
        "the provided tools rather than computing them mentally. "
        "Once all metrics are computed and compared to sector benchmarks, deliver an executive "
        "research memo containing a summary comparison table, balance sheet health review, "
        "and strategic investment takeaways."
    )

    messages = [{"role": "user", "content": user_query}]

    while True:
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=4096,
            system=system_prompt,
            messages=messages,
            tools=tools,
        )

        # Add assistant response turn to history
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"\U0001f527 [Tool Call] {block.name}(args={block.input})")
                    result = dispatch_tool(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result),
                        }
                    )
            messages.append({"role": "user", "content": tool_results})
        elif response.stop_reason == "end_turn":
            # Extract final textual output
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return ""
        else:
            break

## Step 5: Execute Comparative Financial Analysis

We now task the agent with conducting a comparative financial study between **Nvidia (`NVDA`)** and **Apple (`AAPL`)** against their respective sector benchmarks.

In [6]:
sample_query = (
    "Conduct a comprehensive comparative financial analysis between Nvidia (NVDA) and Apple (AAPL). "
    "Retrieve their latest reported financials, calculate their exact valuation and leverage ratios, "
    "compare them to their sector benchmarks, and provide an executive summary with key takeaways."
)

print("Running Financial Research Agent...\n" + "=" * 50)
memo = run_financial_agent(sample_query)
print("\n" + "=" * 50 + "\nFINAL RESEARCH MEMO\n" + "=" * 50)
print(memo)

Running Financial Research Agent...
🔧 [Tool Call] get_company_financials(args={'ticker': 'NVDA'})
🔧 [Tool Call] get_company_financials(args={'ticker': 'AAPL'})
🔧 [Tool Call] calculate_financial_ratios(args={'cash': 34800000000, 'ebitda': 72000000000, 'market_cap': 3100000000000, 'net_income': 63000000000, 'revenue': 126000000000, 'shareholders_equity': 58000000000, 'total_debt': 11000000000})
🔧 [Tool Call] calculate_financial_ratios(args={'cash': 65000000000, 'ebitda': 130000000000, 'market_cap': 3400000000000, 'net_income': 93700000000, 'revenue': 391000000000, 'shareholders_equity': 66700000000, 'total_debt': 106000000000})
🔧 [Tool Call] get_industry_benchmarks(args={'sector': 'Semiconductors'})
🔧 [Tool Call] get_industry_benchmarks(args={'sector': 'Consumer Electronics'})

FINAL RESEARCH MEMO
# Executive Equity Research Memo: NVDA vs. AAPL

### Financial Metrics & Benchmark Comparison

| Metric | Nvidia (NVDA) | Sector Benchmark (Semis) | Apple (AAPL) | Sector Benchmark (CE) |
| :--

## Key Takeaways and Production Patterns

1. **Computational Precision**: By delegating ratio arithmetic (`calculate_financial_ratios`) to Python instead of expecting Claude to perform division within tokens, we eliminate numerical hallucination in mission-critical financial analysis.
2. **Dynamic Tool Chaining**: Claude autonomously decides the sequence: it retrieves company profiles first, identifies the appropriate sector, and subsequently queries sector benchmarks for contextual comparison.
3. **Extensibility**: In enterprise environments, the mock database functions can be swapped with real-time market data providers (e.g., SEC EDGAR, Bloomberg, FactSet, or Financial Modeling Prep) to create an institutional financial copilot.